In [1]:
# Import libraries
from pathlib import Path
import numpy as np
import sys
sys.path.append('../')
import DataLoader as dldr
import AcquisitionFunctions as af
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel, RBF, RationalQuadratic, Matern, ExpSineSquared
from scipy.stats import norm

In [2]:
# Set root directory to load data-points, week & function numbers.
rootDir: Path = Path('..')
weekNbr: int = 4
funcNbr: int = 7

In [3]:
X_inputs_until_prev_week = dldr.load_cumulative_inputs(rootDir, weekNbr - 1, funcNbr)
Y_outputs_until_prev_week = dldr.load_cumulative_outputs(rootDir, weekNbr - 1, funcNbr)

# Print and check some contents from the parsed input & output arrays.
print(f"Function {funcNbr}: until previous week has {len(X_inputs_until_prev_week)} input data-points and {len(Y_outputs_until_prev_week)} output values.")

Week: 3, function 7: Latest input data-point: [0.998333 0.998666 0.999    0.999333 0.999667 1.      ]
Week: 3, function 7: Combined initial & weekly samples contain 33 input data-points.
Week: 3, function 7: Latest output value: 3.4997016155550636e-05
Week: 3, function 7: Combined initial & weekly outputs contain 33 output values, with maximum = 1.3649683044991994.
Function 7: until previous week has 33 input data-points and 33 output values.


In [4]:
y_max_prev_wk = np.max(Y_outputs_until_prev_week)
# Load this week's data
X_this_week_inputs = dldr.load_inputs(rootDir, weekNbr, funcNbr)
Y_this_week_output = dldr.load_output(rootDir, weekNbr, funcNbr)

In [5]:
print(f"Function {funcNbr}: until previous week input data-points: \n{X_inputs_until_prev_week}")
print(f"Function {funcNbr}: until previous week output values: \n{Y_outputs_until_prev_week}")
print(f"Function {funcNbr}: this week's input data-point: {X_this_week_inputs}")
print(f"Function {funcNbr}: this week's output value: {Y_this_week_output}")

Function 7: until previous week input data-points: 
[[2.72623822e-01 3.24495362e-01 8.97108810e-01 8.32951152e-01
  1.54062685e-01 7.95863623e-01]
 [5.43002577e-01 9.24693904e-01 3.41567459e-01 6.46485849e-01
  7.18440327e-01 3.43132664e-01]
 [9.08322480e-02 6.61529382e-01 6.59309106e-02 2.58577008e-01
  9.63452851e-01 6.40265398e-01]
 [1.18866975e-01 6.15054940e-01 9.05816385e-01 8.55300304e-01
  4.13631429e-01 5.85235628e-01]
 [6.30217641e-01 8.38096896e-01 6.80013052e-01 7.31895090e-01
  5.26736715e-01 3.48429213e-01]
 [7.64919173e-01 2.55882917e-01 6.09084224e-01 2.18079042e-01
  3.22942769e-01 9.57936551e-02]
 [5.78955420e-02 4.91672219e-01 2.47422224e-01 2.18118436e-01
  4.20428330e-01 7.30969843e-01]
 [1.95251881e-01 7.92266506e-02 5.54580462e-01 1.70566820e-01
  1.49441764e-02 1.07031710e-01]
 [6.42302982e-01 8.36874547e-01 2.17926915e-02 1.01488010e-01
  6.83070828e-01 6.92416400e-01]
 [7.89942554e-01 1.95545005e-01 5.75623326e-01 7.36591873e-02
  2.59049174e-01 5.10998642e-02

In [6]:
# Combine cumulative inputs until previous week and this week's inputs.
X_inputs_combined = X_inputs_until_prev_week.copy()
X_inputs_combined = np.append(X_inputs_combined, np.array([X_this_week_inputs]), axis=0)

# Combine cumulative outputs until previous week and this week's output.
Y_outputs_combined = Y_outputs_until_prev_week.copy()
Y_outputs_combined = np.append(Y_outputs_combined, np.array([Y_this_week_output]), axis=0)

In [7]:
print(f"Samples aggregated till this week contain {len(X_inputs_combined)} input data-points.")

Samples aggregated till this week contain 34 input data-points.


In [8]:
print(f"Existing (till last week) maximum output: {y_max_prev_wk}, this week's output: {Y_this_week_output}.")

if Y_this_week_output <= y_max_prev_wk:
    print('New output less than existing maximum, do more exploration.')
else:
    print('New output greater than existing maximum, continue exploitation.') 

Existing (till last week) maximum output: 1.3649683044991994, this week's output: 1.8037650755561374.
New output greater than existing maximum, continue exploitation.


In [9]:
# Compute the maximum output value from the aggregated (till this week) data-set.
y_curr_max = np.max(Y_outputs_combined)
print(f"Aggregated (till this week) data-set contain {len(Y_outputs_combined)} output values, with maximum = {y_curr_max}.")

Aggregated (till this week) data-set contain 34 output values, with maximum = 1.8037650755561374.


In [10]:
# Initialize a Gaussian Process surrogate model, which will work as a proxy to
# the (unknown) actual function.
#kernel: Kernel = RBF(length_scale=1.0, length_scale_bounds='fixed')
kernel: Kernel = Matern(length_scale=0.1, length_scale_bounds=(1e-3, 1e3), nu=np.inf)
model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)
model.fit(X_inputs_combined, Y_outputs_combined)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".","Matern(length...e=0.1, nu=inf)"
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",9
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",False
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:

In [11]:
# Apply the model against points on an evaluation grid and capture the predicted values.
x_grid = np.linspace(0, 1, 3000).reshape(-1, 6)
y_pred_means, y_pred_sigmas = model.predict(x_grid, return_std=True)

In [12]:
print(f"x_grid.shape: {x_grid.shape}, y_pred_means.shape: {y_pred_means.shape}, y_pred_sigmas.shape: {y_pred_sigmas.shape}")

x_grid.shape: (500, 6), y_pred_means.shape: (500,), y_pred_sigmas.shape: (500,)


In [13]:
# Initialize an acquisition function to choose the next data-point on the grid to search.
confid_intvl = 0.99
eta = 0.01
acquisition_function = af.ucb(confid_intvl, y_pred_means, y_pred_sigmas)
#acquisition_function = af.prob_improvement(eta, y_pred_means, y_pred_sigmas, y_curr_max)
af_max_idx = np.argmax(acquisition_function)
af_max = acquisition_function[af_max_idx]

x_next = x_grid[af_max_idx]
y_next_mean_pred = y_pred_means[af_max_idx]
y_next_sigma_pred = y_pred_sigmas[af_max_idx]
print('af_max: ', np.max(acquisition_function))
print(f"acquisition_function max: {af_max}, index of max: {af_max_idx}")
print(f"y_pred_means[{af_max_idx}]: {y_next_mean_pred}, y_pred_sigmas[{af_max_idx}]: {y_next_sigma_pred}")

# The relationship between af_max, y_next_mean_pred, y_next_sigma_pred 
# & y_curr_max follows the equation from the function prob_improvement()
# in the module: AcquisitionFunctions. This has been verified separately.

af_max:  1.7332715340103393
acquisition_function max: 1.7332715340103393, index of max: 127
y_pred_means[127]: 1.0503955977246804, y_pred_sigmas[127]: 0.265109157406049


In [14]:
print(f"Current maximum output: {y_curr_max}")
print(f"Next data-point to search: {x_next}.")

Current maximum output: 1.8037650755561374
Next data-point to search: [0.25408469 0.25441814 0.25475158 0.25508503 0.25541847 0.25575192].
